# Terafab Advanced Simulation Lab

This notebook demonstrates the root-level `strategic_simulation/` and `validation_lab/` update package. It uses the existing `terafab_decision_twin` deterministic kernel and adds Monte Carlo, game-theory, reduced-order-model, and validation-readiness workflows.

Boundary: this is screening and validation-readiness analysis, not official Terafab validation or investment advice.

In [ ]:
import json
from pathlib import Path

from terafab_decision_twin.schema import load_scenario
from terafab_decision_twin.engine import run_scenario
from strategic_simulation import run_monte_carlo, analyze_normal_form_game, simulate_reduced_order_model, build_stakeholder_decision_surface
from validation_lab import build_validation_report

ROOT = Path.cwd()
print(ROOT)

## 1. Deterministic kernel run

In [ ]:
scenario = load_scenario(ROOT / 'scenarios' / 'baseline_2026.json')
deterministic = run_scenario(scenario)
deterministic['passed'], deterministic['summary']['total_cost_USD'], deterministic['summary']['legitimacy_margin']

## 2. Monte Carlo uncertainty propagation

In [ ]:
mc_config = json.loads((ROOT / 'strategic_simulation' / 'examples' / 'uncertainty_baseline_2026.json').read_text())
mc_config['runs'] = 50  # increase for deeper analysis
mc = run_monte_carlo(ROOT / 'scenarios' / 'baseline_2026.json', mc_config, retain_runs=False)
mc['passed_probability'], mc['gate_failure_probability']

## 3. Game-theory stakeholder analysis

In [ ]:
game_config = json.loads((ROOT / 'strategic_simulation' / 'examples' / 'strategic_game_2026.json').read_text())
game = analyze_normal_form_game(game_config)
game['pure_strategy_nash_equilibria'], game['conflict_index']

## 4. Reduced-order trajectory

In [ ]:
rom_config = json.loads((ROOT / 'strategic_simulation' / 'examples' / 'rom_transition_2026_2030.json').read_text())
rom = simulate_reduced_order_model(rom_config)
rom['final_state'], rom['diagnostics']

## 5. Validation-readiness report and stakeholder decision surface

In [ ]:
ranges = json.loads((ROOT / 'validation_lab' / 'reference_ranges.json').read_text())
validation = build_validation_report(deterministic, ranges)
surface = build_stakeholder_decision_surface(monte_carlo=mc, game=game, rom=rom, validation=validation)
surface